In [1]:
from __future__ import annotations

import json
import logging
import math
import os
import warnings

import einx
import torch
import torch.nn as nn
from einops import einsum, rearrange
from jaxtyping import Bool, Float, Int
from torch import Tensor

from cs336_basics.nn_utils import softmax

logger = logging.getLogger(__name__)

import gc

In [2]:
class Linear(nn.Module):
    def __init__(self, d_in: int, d_out: int):
        """A linear layer initialized with truncated normal fan-in fan-out.

        Args:
            d_in: int
                The number of input features.
            d_out: int
                The number of output features.
        """

        super().__init__()
        std = math.sqrt(2 / (d_in + d_out))
        self.weight: Float[Tensor, " d_out d_in"] = nn.Parameter(
            nn.init.trunc_normal_(torch.empty(d_out, d_in), std=std, a=-3 * std, b=3 * std), requires_grad=True
        )

    def forward(self, x: Float[Tensor, " ... d_in"]) -> Float[Tensor, " ... d_out"]:
        return einsum(x, self.weight, "... d_in, d_out d_in -> ... d_out")

    def extra_repr(self):
        return f"d_out={self.weight.shape[0]}, d_in={self.weight.shape[1]}"


In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model

        self.q_proj = Linear(self.d_model, self.d_model)
        self.k_proj = Linear(self.d_model, self.d_model)
        self.v_proj = Linear(self.d_model, self.d_model)

    def forward(
        self, x: Float[Tensor, "... seq d_k"]
    ) -> Float[Tensor, "... seq d_v"]:
        *batch_dims, sequence_length, d_model = x.size()
        assert d_model == self.d_model

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        attention_scores = einsum(Q, K, "... query d_k, ... key d_k -> ... query key") / math.sqrt(d_model)

        iota = torch.arange(sequence_length, device=x.device)
        qi = rearrange(iota, "query -> query 1")
        kj = rearrange(iota, "key   -> 1   key")
        causal_mask = qi >= kj
        causal_mask = causal_mask.__getitem__((None,) * len(batch_dims) + (...,))  # Add appropriate leading dimensions
        
        attention_scores = torch.where(causal_mask, attention_scores, float("-inf"))
        

        attention_weights = softmax(attention_scores, dim=-1)

        output = einsum(attention_weights, V, "... query key, ... key d_v -> ... query d_v")

        return output
        

In [4]:
batch = 8

In [5]:
import time


In [12]:
def benchmark_attention_time(d_model, seq_len, use_compiler=False):
        
    model = SelfAttention(d_model).cuda()

    x = torch.randn(
        batch, seq_len, d_model,
        device="cuda",
        requires_grad=True
    )

    if use_compiler:
        model = torch.compile(model)
    # warmup ===============================
    torch.cuda.synchronize()
    # warmup
    for i in range(5):
        model.zero_grad(set_to_none=True)
        x.grad = None
        loss = model(x).sum()
        loss.backward()

    # ================================================
    torch.cuda.synchronize()

    forward_times = []
    backward_times = []

    print('d_model:', d_model, ', seq_len: ', seq_len)
    for i in range(100):
        model.zero_grad(set_to_none=True)
        x.grad = None

        torch.cuda.synchronize()
        start = time.perf_counter()

        loss = model(x).sum()

        torch.cuda.synchronize()
        forward_time = time.perf_counter() - start

        start = time.perf_counter()

        loss.backward()

        torch.cuda.synchronize()
        backward_time = time.perf_counter() - start

        forward_times.append(forward_time)
        backward_times.append(backward_time)

    print(f"Forward:  {sum(forward_times) / 100 * 1000:.3f} ms")
    print(f"Backward: {sum(backward_times) / 100 * 1000:.3f} ms")

    del model, x, loss
    gc.collect()
    
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    return forward_time, backward_time


In [13]:
import torch
import gc

MiB = 1024 ** 2

def check_memory():
    torch.cuda.synchronize()
    print(f"Allocated: {torch.cuda.memory_allocated() / MiB:.2f} MiB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / MiB:.2f} MiB")
    print(f"Cached:    {(torch.cuda.memory_reserved() - torch.cuda.memory_allocated()) / MiB:.2f} MiB")

check_memory()

gc.collect()
torch.cuda.empty_cache()

print("After empty_cache:")
check_memory()


Allocated: 4305.00 MiB
Reserved:  6314.00 MiB
Cached:    2009.00 MiB
After empty_cache:
Allocated: 4305.00 MiB
Reserved:  6314.00 MiB
Cached:    2009.00 MiB


In [ ]:

d_models = [16, 32, 64]
seq_lens = [256, 1024, 4096]

result_dict = {}

for d_model, seq_len in zip(d_models, seq_lens):
    # check_memory()
    forward_time, backward_time = benchmark_attention_time(d_model, seq_len)
    key = f"d_model_{d_model}_seq_len_{seq_len}"
    result_dict[key] = {"forward_time": forward_time, "backward_time": backward_time, "total_time": forward_time + backward_time}

    forward_time, backward_time = benchmark_attention_time(d_model, seq_len, use_compiler=True)
    key = f"d_model_{d_model}_seq_len_{seq_len}_compiler"
    result_dict[key] = {"forward_time": forward_time, "backward_time": backward_time, "total_time": forward_time + backward_time}

d_model: 16 , seq_len:  256
Forward:  1.015 ms
Backward: 1.743 ms


/home/abel/projects/assignment2-systems/.venv/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W1015 23:49:06.478000 74739 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


d_model: 16 , seq_len:  256
Forward:  0.485 ms
Backward: 0.838 ms
d_model: 32 , seq_len:  1024
Forward:  1.879 ms
Backward: 3.527 ms
d_model: 32 , seq_len:  1024
Forward:  0.823 ms
Backward: 1.921 ms


OutOfMemoryError: CUDA out of memory. Tried to allocate 512.00 MiB. GPU 0 has a total capacity of 7.65 GiB of which 171.88 MiB is free. Including non-PyTorch memory, this process has 7.36 GiB memory in use. Of the allocated memory 6.75 GiB is allocated by PyTorch, and 424.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
GiB = 1024 * MiB 

In [ ]:
def benchmark_attention_memory(d_model, seq_len):
    model = SelfAttention(d_model).cuda()
    

    x = torch.randn(
        batch, seq_len, d_model,
        device="cuda",
        requires_grad=True
    )

    # Warmup
    for _ in range(5):
        model.zero_grad(set_to_none=True)
        x.grad = None
        loss = model(x).sum()
        loss.backward()

    torch.cuda.synchronize()

    # Clean up warmup
    del loss
    model.zero_grad(set_to_none=True)
    x.grad = None

    gc.collect()
    torch.cuda.empty_cache()

    # Memory benchmark
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated()

    # Forward
    loss = model(x).sum()

    torch.cuda.synchronize()
    mem_before_backward = torch.cuda.memory_allocated()
    peak_forward = torch.cuda.max_memory_allocated()

    # Backward
    loss.backward()

    torch.cuda.synchronize()
    # mem_after_backward = torch.cuda.memory_allocated()
    peak_total = torch.cuda.max_memory_allocated()

    result = {
        "d_model": d_model,
        "seq_len": seq_len,
        "before_backward_GiB": mem_before_backward / GiB,
        "forward_delta_GiB": (mem_before_backward - mem_before) / GiB,
        "forward_peak_delta_GiB": (peak_forward - mem_before) / GiB,
        "peak_delta_GiB": (peak_total - mem_before) / GiB,
    }

    del loss, x, model
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
import torch
import gc

MiB = 1024 ** 2

def check_memory():
    torch.cuda.synchronize()
    print(f"Allocated: {torch.cuda.memory_allocated() / MiB:.2f} MiB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / MiB:.2f} MiB")
    print(f"Cached:    {(torch.cuda.memory_reserved() - torch.cuda.memory_allocated()) / MiB:.2f} MiB")

check_memory()

gc.collect()
torch.cuda.empty_cache()

print("After empty_cache:")
check_memory()

Allocated: 568.67 MiB
Reserved:  2602.00 MiB
Cached:    2033.33 MiB
After empty_cache:
Allocated: 568.67 MiB
Reserved:  2602.00 MiB
Cached:    2033.33 MiB


In [ ]:

d_models = [16, 32, 64, 128]
seq_lens = [256, 1024, 4096, 8192]

for d_model, seq_len in zip(d_models, seq_lens):
    # check_memory()
    result = benchmark_attention_memory(d_model, seq_len)
    print(result)

{'d_model': 16, 'seq_len': 256, 'before_backward_GiB': 0.5598263740539551, 'forward_delta_GiB': 0.004356861114501953, 'forward_peak_delta_GiB': 0.008264541625976562, 'peak_delta_GiB': 0.012169837951660156}
{'d_model': 32, 'seq_len': 1024, 'before_backward_GiB': 0.622830867767334, 'forward_delta_GiB': 0.0664982795715332, 'forward_peak_delta_GiB': 0.12900543212890625, 'peak_delta_GiB': 0.1914987564086914}
{'d_model': 64, 'seq_len': 4096, 'before_backward_GiB': 1.6026320457458496, 'forward_delta_GiB': 1.0394291877746582, 'forward_peak_delta_GiB': 2.039459228515625, 'peak_delta_GiB': 3.0394296646118164}


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 7.65 GiB of which 813.19 MiB is free. Including non-PyTorch memory, this process has 6.74 GiB memory in use. Of the allocated memory 4.74 GiB is allocated by PyTorch, and 1.80 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)